# 🧹 Banking Credit Risk & Fraud Detection Analytics

## Notebook 02 — Data Cleaning

### Objective

The objective of this notebook is to clean and prepare the Credit Risk and Fraud Detection datasets for exploratory analysis and machine learning.

The cleaning process will focus on:

* Handling duplicate records
* Identifying and treating invalid values
* Handling missing values
* Investigating outliers
* Validating data types
* Checking categorical consistency
* Preserving the original datasets
* Creating clean datasets for downstream analysis

### Important Principle

Data cleaning decisions will be based on evidence from the data rather than automatically removing unusual observations.

The original raw datasets will not be modified. Cleaned versions will be created separately to maintain reproducibility.

## 1. Import Required Libraries

We begin by importing the Python libraries required for data loading, manipulation, numerical analysis, and data-quality checks.

In [2]:
import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Load Raw Datasets

The raw datasets are loaded from the `data` directory.

We use relative paths so that the project remains portable across different environments such as local machines, GitHub repositories, Docker containers, and cloud deployment environments.

In [3]:
credit_path = "../data/credit_risk/credit_risk_dataset.csv"
fraud_path = "../data/fraud/creditcard.csv"

credit_df = pd.read_csv(credit_path)
fraud_df = pd.read_csv(fraud_path)

print("Credit Risk Shape :", credit_df.shape)
print("Fraud Detection Shape :", fraud_df.shape)

Credit Risk Shape : (32581, 12)
Fraud Detection Shape : (284807, 31)


## 3. Preserve Raw Data

The original DataFrames are treated as raw data and will not be overwritten.

Copies will be created for cleaning so that:

- The original data remains available for comparison.
- Cleaning decisions can be reversed.
- Data-processing steps remain reproducible.
- Potential data-quality issues can be investigated without losing the original observations.

In [4]:
credit_clean = credit_df.copy()
fraud_clean = fraud_df.copy()

print("Working copies created successfully!")

Working copies created successfully!


## 4. Duplicate Record Handling

Duplicate records can cause certain observations to receive excessive representation during analysis and model training.

The initial data audit identified:

- 165 duplicate records in the Credit Risk dataset.
- 1,081 duplicate records in the Fraud Detection dataset.

Since these are exact duplicate rows across all available columns, they will be removed from the working copies.

The raw datasets remain unchanged.

In [5]:
credit_duplicates = credit_clean.duplicated().sum()
fraud_duplicates = fraud_clean.duplicated().sum()

print("Credit Risk duplicates :", credit_duplicates)
print("Fraud duplicates       :", fraud_duplicates)

Credit Risk duplicates : 165
Fraud duplicates       : 1081


In [6]:
credit_clean = credit_clean.drop_duplicates().reset_index(drop=True)
fraud_clean = fraud_clean.drop_duplicates().reset_index(drop=True)

print("Credit Risk shape after duplicate removal :", credit_clean.shape)
print("Fraud shape after duplicate removal       :", fraud_clean.shape)

Credit Risk shape after duplicate removal : (32416, 12)
Fraud shape after duplicate removal       : (283726, 31)


### 🔎 Finding

Exact duplicate records were removed from the working datasets.

- Credit Risk: 165 duplicate rows removed.
- Fraud Detection: 1,081 duplicate rows removed.

The original raw datasets were preserved, ensuring that the cleaning process remains reproducible.

## 5. Handle Invalid Credit Risk Values

The initial data-quality investigation identified clearly suspicious values:

- Applicant ages greater than 100 years.
- Employment length of 123 years.

These values are not considered realistic for the intended banking use case.

Instead of deleting the complete records, the invalid feature values will be converted to `NaN`.

This approach preserves the remaining information in the customer records while allowing the invalid values to be handled during the preprocessing stage.

In [7]:
credit_clean.loc[
    credit_clean["person_age"] > 100,
    "person_age"
] = np.nan

credit_clean.loc[
    credit_clean["person_emp_length"] > 50,
    "person_emp_length"
] = np.nan

print("Invalid age and employment-length values handled.")

Invalid age and employment-length values handled.


## 6. Validate Invalid-Value Treatment

After replacing the identified invalid values with missing values, we verify that no unrealistic age or employment-length values remain.

In [8]:
print("Age > 100:")
print(credit_clean[credit_clean["person_age"] > 100])

print("\nEmployment length > 50:")
print(credit_clean[credit_clean["person_emp_length"] > 50])

Age > 100:
Empty DataFrame
Columns: [person_age, person_income, person_home_ownership, person_emp_length, loan_intent, loan_grade, loan_amnt, loan_int_rate, loan_status, loan_percent_income, cb_person_default_on_file, cb_person_cred_hist_length]
Index: []

Employment length > 50:
Empty DataFrame
Columns: [person_age, person_income, person_home_ownership, person_emp_length, loan_intent, loan_grade, loan_amnt, loan_int_rate, loan_status, loan_percent_income, cb_person_default_on_file, cb_person_cred_hist_length]
Index: []


## 7. Missing Value Analysis

Missing values were identified during the initial data-understanding stage.

The Credit Risk dataset contains missing values in:

- `person_emp_length`
- `loan_int_rate`

The Fraud Detection dataset originally contained no missing values.

Before choosing an imputation strategy, we will examine the missing-value percentage and the distribution of the affected variables.

### Cleaning Principle

Missing values should not automatically be replaced with the mean or median.

The appropriate strategy depends on:

- The variable's distribution
- The meaning of the variable
- The amount of missing data
- Potential relationships with other variables
- The requirements of the machine-learning model

The final production preprocessing pipeline will perform imputation using parameters learned from the training data only, preventing data leakage.

In [9]:
credit_missing = credit_clean.isnull().sum()
credit_missing_percentage = (
    credit_clean.isnull().mean() * 100
)

missing_summary = pd.DataFrame({
    "Missing_Count": credit_missing,
    "Missing_Percentage": credit_missing_percentage
})

missing_summary = missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values(
    "Missing_Percentage",
    ascending=False
)

missing_summary

,Missing_Count,Missing_Percentage
loan_int_rate,3095,9.547754
person_emp_length,889,2.742473
person_age,5,0.015424


## 8. Investigating Missing Interest Rates

`loan_int_rate` contains a relatively high proportion of missing values.

Because interest rate is a numerical variable, we need to understand its distribution before deciding whether mean, median, or another approach is appropriate.

The median is generally more robust than the mean when a numerical variable contains skewness or extreme observations.

In [10]:
credit_clean["loan_int_rate"].describe()

count    29321.000000
mean        11.017265
std          3.241680
min          5.420000
25%          7.900000
50%         10.990000
75%         13.470000
max         23.220000
Name: loan_int_rate, dtype: float64

In [12]:
credit_clean["loan_int_rate"].skew()

np.float64(0.2070016541005385)

## 9. Investigating Missing Employment Length

`person_emp_length` contains missing values, including values that were identified as invalid during the data-quality investigation.

After converting the clearly invalid employment-length value of 123 years to `NaN`, the feature now contains both originally missing values and invalid values requiring treatment.

We will examine its distribution before selecting an imputation strategy.

In [11]:
credit_clean["person_emp_length"].describe()

count    31527.000000
mean         4.783011
std          4.037282
min          0.000000
25%          2.000000
50%          4.000000
75%          7.000000
max         41.000000
Name: person_emp_length, dtype: float64

In [12]:
credit_clean["person_emp_length"].skew()

np.float64(1.2495259321825027)

## 10. Determine Imputation Values

Based on the distribution analysis, median imputation will be used for the missing numerical values.

The median is preferred because:

- `loan_int_rate` contains missing observations and may contain some distributional variation.
- `person_emp_length` is positively skewed.
- The median is less sensitive to extreme values than the mean.

For this exploratory cleaning notebook, the median values are calculated from the cleaned working dataset.

In the final machine-learning pipeline, these values will be learned from the training data only to prevent data leakage.

In [13]:
emp_length_median = credit_clean["person_emp_length"].median()
interest_rate_median = credit_clean["loan_int_rate"].median()

print("Employment length median :", emp_length_median)
print("Interest rate median     :", interest_rate_median)

Employment length median : 4.0
Interest rate median     : 10.99


## 11. Apply Missing-Value Imputation

Based on the distribution analysis, median imputation will be applied to the missing values in the Credit Risk dataset.

The selected values are:

- `person_emp_length` → 4.0 years
- `loan_int_rate` → 10.99%

Median imputation preserves the number of observations and is less sensitive to extreme values than mean imputation.

The imputation is applied only to the working copy. The original raw dataset remains unchanged.`m

In [14]:
credit_clean["person_emp_length"] = credit_clean["person_emp_length"].fillna(
    emp_length_median
)

credit_clean["loan_int_rate"] = credit_clean["loan_int_rate"].fillna(
    interest_rate_median
)

print("Missing values imputed successfully.")

Missing values imputed successfully.


## 12. Validate Missing-Value Treatment

After imputation, we verify that the previously missing values in `person_emp_length` and `loan_int_rate` have been handled successfully.

In [15]:
credit_clean[
    ["person_emp_length", "loan_int_rate"]
].isnull().sum()

person_emp_length    0
loan_int_rate        0
dtype: int64

### 🔎 Finding

The missing values in `person_emp_length` and `loan_int_rate` have been successfully imputed using their respective median values.

No missing values remain in these two variables.

The original raw dataset has not been modified.

## 13. Fraud Dataset Missing-Value Validation

The initial data audit found no missing values in the Fraud Detection dataset.

Therefore, no missing-value imputation is required for the fraud dataset.

We perform a final validation to confirm that the cleaned working dataset still contains no missing values.

In [16]:
fraud_clean.isnull().sum().sum()

np.int64(0)

### 🔎 Finding

The Fraud Detection dataset contains no missing values, so no missing-value treatment was required.

The dataset is complete with respect to missing-value checks.

## 14. Outlier Analysis

Outliers are observations that are unusually far from the majority of the data.

Outliers can occur for two reasons:

1. **Data-quality issues** — impossible or incorrectly recorded values.
2. **Legitimate extreme observations** — genuine customers whose characteristics are unusual.

Therefore, extreme values will not be automatically removed.

The analysis will distinguish between invalid observations and legitimate extreme observations.

For the Credit Risk dataset, particular attention will be given to:

- `person_age`
- `person_income`
- `person_emp_length`
- `loan_amnt`
- `loan_percent_income`

The goal is to preserve valid business information while preventing clearly invalid values from negatively affecting downstream analysis and modeling.

### 14.1 Applicant Age

Applicant age was previously investigated during the data-understanding stage.

Values above 100 years were identified as unrealistic for this dataset and converted to missing values.

The cleaned distribution is examined below to verify the resulting range.

In [17]:
credit_clean["person_age"].describe()

count    32411.000000
mean        27.730369
std          6.210448
min         20.000000
25%         23.000000
50%         26.000000
75%         30.000000
max         94.000000
Name: person_age, dtype: float64

In [18]:
credit_clean["person_age"].isnull().sum()

np.int64(5)

### 14.2 Age Imputation

Five applicant records contained unrealistic ages greater than 100 years.

Rather than removing the complete records, these invalid age values were converted to missing values.

The median age will be used to replace these invalid values because the median is robust to extreme observations and preserves the affected records.

The production preprocessing pipeline will learn the median from the training data only.`````

In [19]:
age_median = credit_clean["person_age"].median()

print("Median age:", age_median)

Median age: 26.0


## 15. Apply Age Imputation

The five invalid age values identified during data-quality analysis were converted to missing values.

The median applicant age is 26 years.

These missing age values will now be replaced with the median age of 26 years rather than removing the complete customer records.

In [20]:
credit_clean["person_age"] = credit_clean["person_age"].fillna(age_median)

print("Age imputation completed.")

Age imputation completed.


## 16. Validate Age Cleaning

The age-cleaning process is validated by checking:

- Whether any missing age values remain.
- Whether any applicant age is greater than 100.
- The final minimum and maximum age.

In [21]:
print("Missing ages:", credit_clean["person_age"].isnull().sum())
print("Ages above 100:", (credit_clean["person_age"] > 100).sum())
print("Minimum age:", credit_clean["person_age"].min())
print("Maximum age:", credit_clean["person_age"].max())

Missing ages: 0
Ages above 100: 0
Minimum age: 20.0
Maximum age: 94.0


### 🔎 Finding

The five unrealistic age observations were successfully treated and replaced using the median age of 26 years.

After cleaning, no missing age values or ages above 100 remain.

The affected customer records were retained rather than being completely removed, preserving the available information for downstream analysis.

## 17. Income Outlier Analysis

`person_income` contains a wide range of values, with a maximum value of 6,000,000.

A high income is not automatically an invalid observation because high-income customers can legitimately exist.

Therefore, statistical outlier detection will be used to identify unusually large observations, followed by business and data-quality investigation.

The Interquartile Range (IQR) method is used as an initial statistical technique.

### IQR Method

The IQR is calculated as:

**IQR = Q3 − Q1**

The upper outlier boundary is:

**Upper Bound = Q3 + 1.5 × IQR**

Observations above this boundary will be flagged for investigation rather than automatically deleted.

In [22]:
Q1 = credit_clean["person_income"].quantile(0.25)
Q3 = credit_clean["person_income"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

Q1: 38542.0
Q3: 79218.0
IQR: 40676.0
Lower Bound: -22472.0
Upper Bound: 140232.0


### 17.1 Statistical Income Outliers

The IQR upper bound will be used to determine how many income observations are statistically classified as outliers.

These observations will be investigated before any transformation or removal decision is made.

In [23]:
income_outliers = credit_clean[
    credit_clean["person_income"] > upper_bound
]

print("Number of income outliers:", len(income_outliers))
print(
    "Percentage of income outliers:",
    round(len(income_outliers) / len(credit_clean) * 100, 2),
    "%"
)

Number of income outliers: 1478
Percentage of income outliers: 4.56 %


### 17.2 Investigate Income Outliers

The identified income outliers are examined alongside age, employment length, loan amount, and loan status.

This helps determine whether the extreme income values appear to be invalid records or legitimate high-income applicants.

In [24]:
income_outliers[
    [
        "person_age",
        "person_income",
        "person_emp_length",
        "loan_amnt",
        "loan_percent_income",
        "loan_status"
    ]
].sort_values(
    "person_income",
    ascending=False
).head(20)

,person_age,person_income,person_emp_length,loan_amnt,loan_percent_income,loan_status
32132,26.0,6000000,12.0,5000,0.00,0
29890,42.0,2039784,0.0,8450,0.00,0
32381,60.0,1900000,5.0,1500,0.00,0
32332,63.0,1782000,13.0,12025,0.01,0
31765,44.0,1440000,7.0,6400,0.00,0
31763,47.0,1362000,9.0,6600,0.00,0
28961,40.0,1200000,1.0,10000,0.01,0
28960,36.0,1200000,16.0,10000,0.01,0
17711,32.0,1200000,1.0,12000,0.01,0
17712,34.0,948000,18.0,2000,0.00,0


## 18. Income Distribution and Transformation Analysis

The IQR method identified 1,478 statistical income outliers, representing approximately 4.56% of the cleaned dataset.

However, statistical outliers are not necessarily invalid observations.

Several identified applicants have high incomes combined with relatively small loan amounts, which may represent legitimate financial profiles.

Therefore, the income observations will not be removed solely because they are classified as IQR outliers.

Instead, the distribution will be evaluated for skewness to determine whether a transformation would be appropriate for machine learning.

In [25]:
print("Income skewness:", credit_clean["person_income"].skew())

Income skewness: 32.95874828815014


In [26]:
print(
    credit_clean["person_income"].describe(
        percentiles=[0.90, 0.95, 0.99, 0.995]
    )
)

count    3.241600e+04
mean     6.609164e+04
std      6.201558e+04
min      4.000000e+03
50%      5.500000e+04
90%      1.102200e+05
95%      1.380000e+05
99%      2.250000e+05
99.5%    3.000000e+05
max      6.000000e+06
Name: person_income, dtype: float64


## 19. Log Transformation Investigation

`person_income` has a wide right-skewed distribution.

A logarithmic transformation can reduce the influence of extreme values while preserving the relative ordering of applicants.

The transformation will be investigated before deciding whether it should become part of the final feature-engineering pipeline.

The original income variable will be preserved.

In [27]:
income_log = np.log1p(credit_clean["person_income"])

print("Original skewness:", credit_clean["person_income"].skew())
print("Log-transformed skewness:", income_log.skew())

Original skewness: 32.95874828815014
Log-transformed skewness: 0.15776472961753327


## 20. Income Transformation Decision

The original `person_income` variable has extremely high positive skewness:

- Original skewness: **32.96**
- Log-transformed skewness: **0.16**

The `log1p()` transformation substantially reduces the right skew while preserving the relative ordering of income values.

Because the identified high-income observations may represent legitimate applicants, they will not be removed simply because they are statistical outliers.

Instead, a log-transformed income feature will be retained for downstream machine-learning analysis.

The original `person_income` variable will also be preserved so that the original business meaning remains available.

### Decision

`person_income` → preserve original + create `person_income_log`

This allows the machine-learning stage to evaluate whether the transformed feature improves model performance.

In [28]:
credit_clean["person_income_log"] = np.log1p(
    credit_clean["person_income"]
)

print("Log-transformed income feature created.")

Log-transformed income feature created.


In [29]:
credit_clean[
    ["person_income", "person_income_log"]
].head()

,person_income,person_income_log
0,59000,10.985310
1,9600,9.169623
2,9600,9.169623
3,65500,11.089821
4,54400,10.904138


## 21. Validate Income Transformation

The newly created `person_income_log` feature is validated by comparing its statistical distribution with the original income variable.

The objective is to confirm that the transformation has substantially reduced the extreme right skew while retaining all observations.

In [30]:
print("Original income skewness:",
      credit_clean["person_income"].skew())

print("Log income skewness:",
      credit_clean["person_income_log"].skew())

Original income skewness: 32.95874828815014
Log income skewness: 0.15776472961753327


## 22. Loan-to-Income Ratio Analysis

The `loan_percent_income` variable represents the proportion of an applicant's income associated with the requested loan.

The initial data audit showed values ranging from 0.00 to 0.83.

Unlike clearly invalid values such as an applicant age of 144 years, a high loan-to-income ratio is not necessarily a data-quality error.

A high ratio may represent a legitimate financial situation and may also contain useful predictive information for credit-risk modeling.

Therefore, extreme values will be investigated rather than automatically removed.

In [31]:
credit_clean["loan_percent_income"].describe(
    percentiles=[0.90, 0.95, 0.99, 0.995]
)

count    32416.000000
mean         0.170250
std          0.106812
min          0.000000
50%          0.150000
90%          0.320000
95%          0.380000
99%          0.500000
99.5%        0.530000
max          0.830000
Name: loan_percent_income, dtype: float64

In [32]:
print(
    "Loan percent income skewness:",
    credit_clean["loan_percent_income"].skew()
)

Loan percent income skewness: 1.0638113566804195


### 22.1 Investigating High Loan-to-Income Ratios

The highest loan-to-income observations are examined to determine whether they appear to be valid financial profiles.

We will compare the ratio with income, loan amount, age, and loan status rather than removing observations based only on a statistical threshold.

In [33]:
credit_clean[
    [
        "person_age",
        "person_income",
        "loan_amnt",
        "loan_percent_income",
        "loan_status"
    ]
].sort_values(
    "loan_percent_income",
    ascending=False
).head(20)

,person_age,person_income,loan_amnt,loan_percent_income,loan_status
640,22.0,20000,16600,0.83,0
23605,32.0,12000,9325,0.78,1
577,26.0,26000,20050,0.77,1
571,21.0,19500,15000,0.77,1
18081,30.0,32004,24250,0.76,1
460,24.0,18000,13000,0.72,1
2452,25.0,32004,22750,0.71,0
27756,33.0,10080,7200,0.71,1
10007,21.0,13000,9250,0.71,1
18550,28.0,24000,16750,0.70,1


## 23. Loan-to-Income Ratio — Cleaning Decision

The `loan_percent_income` variable represents the proportion of an applicant's income associated with the requested loan.

Although high values may appear statistically unusual, they can represent legitimate financial situations and may contain important predictive information for credit-risk modeling.

No missing values or clearly invalid values were identified in this feature.

### Decision

`loan_percent_income` will be retained without outlier removal, capping, or transformation.

The feature will be evaluated during exploratory analysis and machine-learning model development to determine its relationship with loan default.

## 24. Loan Amount Validation

`loan_amnt` represents the amount of money requested by the applicant.

The initial data audit showed values ranging from 500 to 35,000.

We will verify the distribution and check for unusual values before deciding whether any treatment is required.

In [34]:
credit_clean["loan_amnt"].describe(
    percentiles=[0.90, 0.95, 0.99, 0.995]
)

count    32416.000000
mean      9593.845632
std       6322.730241
min        500.000000
50%       8000.000000
90%      19037.500000
95%      24000.000000
99%      29800.000000
99.5%    35000.000000
max      35000.000000
Name: loan_amnt, dtype: float64

In [35]:
print(
    "Loan amount skewness:",
    credit_clean["loan_amnt"].skew()
)

Loan amount skewness: 1.191944385120234


In [36]:
credit_clean[
    ["person_income", "loan_amnt", "loan_percent_income", "loan_status"]
].sort_values(
    "loan_amnt",
    ascending=False
).head(20)

,person_income,loan_amnt,loan_percent_income,loan_status
16,120000,35000,0.29,0
32413,76000,35000,0.46,1
14,115000,35000,0.30,0
17,92111,35000,0.32,1
0,59000,35000,0.59,1
18,113000,35000,0.31,1
20,162500,35000,0.22,0
12,95000,35000,0.37,1
13,108160,35000,0.32,1
15296,130000,35000,0.27,0


## 25. Credit Risk Cleaning Validation

The Credit Risk dataset is validated after the cleaning operations.

The validation checks:

- Remaining missing values
- Remaining duplicate records
- Invalid applicant ages
- Invalid employment lengths
- Dataset dimensions
- Data types

In [37]:
print("Shape:", credit_clean.shape)

print("\nMissing values:")
print(credit_clean.isnull().sum())

print("\nDuplicate rows:")
print(credit_clean.duplicated().sum())

print("\nMaximum age:")
print(credit_clean["person_age"].max())

print("\nMaximum employment length:")
print(credit_clean["person_emp_length"].max())

Shape: (32416, 13)

Missing values:
person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_status                   0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
person_income_log             0
dtype: int64

Duplicate rows:
0

Maximum age:
94.0

Maximum employment length:
41.0


## 26. Fraud Dataset Cleaning Validation

The Fraud Detection dataset is validated after duplicate removal.

The initial audit found:

- No missing values.
- 1,081 exact duplicate records.

Since no invalid numerical values requiring correction were identified during the initial audit, the primary cleaning operation for this dataset was duplicate removal.

In [38]:
print("Shape:", fraud_clean.shape)

print("\nTotal missing values:")
print(fraud_clean.isnull().sum().sum())

print("\nDuplicate rows:")
print(fraud_clean.duplicated().sum())

print("\nTarget distribution:")
print(fraud_clean["Class"].value_counts())

Shape: (283726, 31)

Total missing values:
0

Duplicate rows:
0

Target distribution:
Class
0    283253
1       473
Name: count, dtype: int64


# 27. Data Cleaning Summary

## Credit Risk Dataset

| Issue | Action |
|---|---|
| Missing `person_emp_length` | Median imputation |
| Missing `loan_int_rate` | Median imputation |
| Invalid age > 100 | Converted to `NaN`, then median imputed |
| Employment length = 123 | Converted to `NaN`, then median imputed |
| Duplicate rows | Removed |
| Extreme income values | Preserved |
| Income skewness | Log-transformed feature created |
| `loan_percent_income` | Preserved |

## Fraud Detection Dataset

| Issue | Action |
|---|---|
| Missing values | None found |
| Duplicate rows | Removed |
| Extreme class imbalance | Preserved for appropriate modeling techniques |
| `V1`–`V28` | Preserved as anonymized numerical features |

### Important Note

The raw datasets were never modified. All cleaning operations were performed on working copies.

The final machine-learning implementation will use reproducible preprocessing pipelines and will learn preprocessing parameters from training data only to prevent data leakage.

## 28. Export Cleaned Datasets

The cleaned datasets are exported to the `data/processed` directory.

Keeping processed datasets separate from raw data improves reproducibility and prevents accidental modification of the original datasets.

In [39]:
import os

processed_path = "../data/processed"

os.makedirs(processed_path, exist_ok=True)

credit_clean.to_csv(
    f"{processed_path}/credit_risk_cleaned.csv",
    index=False
)

fraud_clean.to_csv(
    f"{processed_path}/fraud_cleaned.csv",
    index=False
)

print("Cleaned datasets exported successfully!")

Cleaned datasets exported successfully!


# 29. Conclusion

The data-cleaning stage has been completed for both Credit Risk and Fraud Detection datasets.

The Credit Risk dataset was cleaned by addressing missing values, invalid age and employment-length observations, duplicate records, and extreme income distribution.

A log-transformed income feature was created to reduce the extreme positive skew while preserving the original income variable.

The Fraud Detection dataset contained no missing values. Duplicate transactions were removed while the severe class imbalance was preserved because it represents an important characteristic of the fraud-detection problem.

The cleaned datasets have been exported to the `data/processed` directory.

### Next Step

➡️ **03_Exploratory_Data_Analysis.ipynb**

The next stage will explore relationships between customer characteristics, loan characteristics, default behavior, transaction characteristics, and fraud patterns using statistical summaries and visualizations.